In [1]:
import numpy as np
import matplotlib.pyplot as plt

def run_experiment(observations=1000, eta=0.02, num_epochs=100, custom_formula=False):
    xs = np.random.uniform(low=-10, high=10, size=(observations, 1))
    zs = np.random.uniform(low=-10, high=10, size=(observations, 1))
    inputs = np.column_stack((xs, zs))

    noise = np.random.uniform(low=-1, high=1, size=(observations, 1))

    if custom_formula:
        targets = 13 * xs + 7 * zs - 12 + noise
        expected_w = [13.0, 7.0]
        expected_b = -12.0
    else:
        targets = 2 * xs - 3 * zs + 5 + noise
        expected_w = [2.0, -3.0]
        expected_b = 5.0

    init_range = 0.1
    weights = np.random.uniform(low=-init_range, high=init_range, size=(2, 1))
    biases = np.random.uniform(low=-init_range, high=init_range, size=1)

    loss_history = []

    targets = targets.reshape(observations, 1)
    for i in range(num_epochs):
        outputs = np.dot(inputs, weights) + biases
        deltas = outputs - targets

        loss = np.sum(deltas ** 2) / 2 / observations
        loss_history.append(loss)

        if np.isnan(loss) or np.isinf(loss) or (i > 0 and loss > loss_history[0] * 1e5):
            return "Eksplozja gradientu", None, None, None

        deltas_scaled = deltas / observations
        weights -= eta * np.dot(inputs.T, deltas_scaled)
        biases -= eta * np.sum(deltas_scaled)

    return loss_history[-1], weights, biases, loss_history


print("=== 1. ZMIANA ILOŚCI PRÓBEK ===")
for obs in [1000, 10000, 100000]:
    loss, w, b, _ = run_experiment(observations=obs, eta=0.02)
    print(f"Próbki: {obs:7} | Loss końcowy: {loss:.4f} | Wagi: [{w[0][0]:.4f}, {w[1][0]:.4f}] | Bias: {b[0]:.4f}")

print("\n=== 2. ZMIANA WSPÓŁCZYNNIKA UCZENIA SIĘ (ETA) ===")
etas = [0.0001, 0.001, 0.1, 1.0]
for e in etas:
    result = run_experiment(observations=1000, eta=e, num_epochs=100)
    if isinstance(result[0], str):
        print(f"Eta: {e:6} -> Wynik: {result[0]} (Model nie zbiega się)")
    else:
        loss, w, b, _ = result
        print(f"Eta: {e:6} | Loss końcowy: {loss:.4f} | Wagi: [{w[0][0]:.4f}, {w[1][0]:.4f}] | Bias: {b[0]:.4f}")

print("\n=== 3. ZMIANA TARGETS (13*xs + 7*zs - 12) ===")
loss, w, b, _ = run_experiment(observations=1000, eta=0.02, num_epochs=200, custom_formula=True)
print(f"Oczekiwane: w1=13, w2=7, b=-12")
print(f"Uzyskane:   w1={w[0][0]:.4f}, w2={w[1][0]:.4f}, b={b[0]:.4f} | Loss: {loss:.4f}")

=== 1. ZMIANA ILOŚCI PRÓBEK ===
Próbki:    1000 | Loss końcowy: 0.4137 | Wagi: [2.0029, -3.0006] | Bias: 4.3295
Próbki:   10000 | Loss końcowy: 0.4014 | Wagi: [2.0011, -2.9993] | Bias: 4.3241
Próbki:  100000 | Loss końcowy: 0.3909 | Wagi: [1.9996, -3.0001] | Bias: 4.3443

=== 2. ZMIANA WSPÓŁCZYNNIKA UCZENIA SIĘ (ETA) ===
Eta: 0.0001 | Loss końcowy: 126.2380 | Wagi: [0.6075, -0.7857] | Bias: 0.0655
Eta:  0.001 | Loss końcowy: 10.4127 | Wagi: [1.9012, -2.8943] | Bias: 0.5591
Eta:    0.1 -> Wynik: Eksplozja gradientu (Model nie zbiega się)
Eta:    1.0 -> Wynik: Eksplozja gradientu (Model nie zbiega się)

=== 3. ZMIANA RECEPTURY TARGETS (13*xs + 7*zs - 12) ===
Oczekiwane: w1=13, w2=7, b=-12
Uzyskane:   w1=13.0004, w2=7.0010, b=-11.7671 | Loss: 0.1888
